In [4]:
using LowLevelFEM

structured_rect_mesh(lx=1.0, ly=0.04, y0=-0.02, n=2, order=2)
mat = Material("body"; E=1e6, ν=0.3, ρ=1000.0)
U = Field([mat]; type=:VectorField, dim=2, fieldName=:u)
b, D2 = 0.01, D(:PlaneStress, mat)
support = [BoundaryCondition("left"; field=U, ux=0.0, uy=0.0)]
u0 = initialDisplacement(U, "body"; ux=0.0, uy=0.0)
v0 = initialVelocity(U, "body"; vx=0.0, vy=(x,y,z)->0.25*x^2*(3-x)/2)
M = b*mat.ρ*∫(U⋅U)
K = b*∫(SymGrad(U)⋅D2⋅SymGrad(U))
#K = ∫(SymGrad(U)⋅D2⋅SymGrad(U); weight=b, csc_matrix=build_csc_pattern(U, U))
B0 = [1 0 0 0; 0 0 0 1; 0 1 1 0]

function secant(A, state)
    H = ∇(expandTo3D(state.fields[1]))
    F = tensorToMatrix(unitTensor(H) + H)
    B = [F[1,1] 0 F[2,1] 0; 0 F[1,2] 0 F[2,2];
         F[1,2] F[1,1] F[2,2] F[2,1]]
    b*∫(Grad(U)⋅B'⋅D2⋅((B+B0)/2)⋅Grad(U); gauss=4)
    #fill!(A.A.nzval, 0.0)
    #∫(Grad(U)⋅B'⋅D2⋅((B+B0)/2)⋅Grad(U); gauss=4, weight=b, csc_matrix=A.A)
end

@time u, v = HHT(K, M, 0*u0, u0, v0, 401, 0.02;
    support, α=-0.05, modifyA0=secant,
    nonlinearoptions=(maxiter=50, abstol=1e-9),
    copyOperators=true)
showDoFResults(u; name="Nagy lengés — HHT", factor=1.0, visible=true)

 71.586168 seconds (287.89 M allocations: 49.690 GiB, 14.27% gc time, 6.88% compilation time: 32% of which was recompilation)


0

In [5]:
openPostProcessor()